## Seq2Seq English→Spanish

In [ ]:
import random
import re
import json
import time
from pathlib import Path
from collections import Counter
from typing import List, Tuple, Dict, Any, Optional

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import sacrebleu
import models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

DATA_PATH = Path("spa.txt")

SAMPLE_N = 10_000

TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10

LOWERCASE = True
MIN_FREQ = 2
MAX_VOCAB_SRC = 20_000
MAX_VOCAB_TGT = 20_000

MAX_LEN_SRC = 50
MAX_LEN_TGT = 50

BATCH_SIZE = 64
NUM_WORKERS = 0

EPOCHS = 50

MODELS_DIR = Path("models")
RUNS_DIR = Path("runs")
OUT_DIR = Path("outputs")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

EMB_SRC = 256
EMB_TGT = 256
HIDDEN  = 512
NUM_LAYERS = 1
DROPOUT = 0.1

DEVICE: mps


## Load dataset + split

In [ ]:
def load_pairs(path: Path) -> List[Tuple[str, str]]:
    pairs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) < 2:
                continue
            en, es = parts[0], parts[1]
            pairs.append((en, es))
    return pairs

def split_pairs(
    pairs: List[Tuple[str, str]],
    sample_n: Optional[int] = None,
    train_ratio: float = 0.8,
    val_ratio: float = 0.1,
    seed: int = 42
):
    rng = random.Random(seed)
    pairs = pairs[:]
    rng.shuffle(pairs)

    if sample_n is not None:
        pairs = pairs[:sample_n]

    n = len(pairs)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train = pairs[:n_train]
    val   = pairs[n_train:n_train + n_val]
    test  = pairs[n_train + n_val:]

    return train, val, test

pairs = load_pairs(DATA_PATH)
train_pairs, val_pairs, test_pairs = split_pairs(
    pairs,
    sample_n=SAMPLE_N,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    seed=SEED
)

print("Total:", len(pairs), "| Using:", len(train_pairs)+len(val_pairs)+len(test_pairs))
print("Train:", len(train_pairs), "Val:", len(val_pairs), "Test:", len(test_pairs))


Total: 118964 | Using: 10000
Train: 8000 Val: 1000 Test: 1000


## Tokenization + vocab + Dataset/DataLoader

In [ ]:
PAD = "<pad>"
UNK = "<unk>"
BOS = "<bos>"
EOS = "<eos>"
SPECIAL_TOKENS = [PAD, UNK, BOS, EOS]

def normalize(text: str, lowercase: bool = True) -> str:
    text = text.strip()
    if lowercase:
        text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text

def tokenize(text: str) -> List[str]:
    text = re.sub(r"([.,!?;:()\"'])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split() if text else []

def build_vocab_from_pairs(pairs, side: str, min_freq: int = 2, max_vocab: Optional[int] = None) -> Dict[str, Any]:
    assert side in {"src", "tgt"}
    idx = 0 if side == "src" else 1

    counter = Counter()
    for en, es in pairs:
        text = normalize(en if idx == 0 else es, LOWERCASE)
        toks = tokenize(text)
        counter.update(toks)

    itos = list(SPECIAL_TOKENS)
    for w, c in counter.most_common():
        if c < min_freq:
            break
        if max_vocab is not None and len(itos) >= max_vocab:
            break
        if w not in SPECIAL_TOKENS:
            itos.append(w)

    stoi = {w: i for i, w in enumerate(itos)}
    return {"itos": itos, "stoi": stoi}

def ids_to_sentence(ids: List[int], itos: List[str], stop_at_eos: bool = True) -> str:
    toks = []
    for i in ids:
        tok = itos[int(i)]
        if tok in ("<pad>", "<bos>"):
            continue
        if stop_at_eos and tok == "<eos>":
            break
        toks.append(tok)
    sent = " ".join(toks)
    sent = re.sub(r"\s+([.,!?;:])", r"\1", sent)
    sent = re.sub(r"\s+", " ", sent).strip()
    return sent

def batch_ids_to_sentences(batch_ids: torch.Tensor, itos: List[str]) -> List[str]:
    return [ids_to_sentence(row.tolist(), itos, stop_at_eos=True) for row in batch_ids]

src_vocab = build_vocab_from_pairs(train_pairs, "src", min_freq=MIN_FREQ, max_vocab=MAX_VOCAB_SRC)
tgt_vocab = build_vocab_from_pairs(train_pairs, "tgt", min_freq=MIN_FREQ, max_vocab=MAX_VOCAB_TGT)

pad_id_src = src_vocab["stoi"][PAD]
pad_id_tgt = tgt_vocab["stoi"][PAD]
bos_id_tgt = tgt_vocab["stoi"][BOS]
eos_id_tgt = tgt_vocab["stoi"][EOS]

def encode(text: str, vocab: Dict[str, Any], add_bos_eos: bool = False, max_len: int = 50) -> List[int]:
    text = normalize(text, LOWERCASE)
    toks = tokenize(text)
    if add_bos_eos:
        toks = [BOS] + toks + [EOS]
    toks = toks[:max_len]
    ids = [vocab["stoi"].get(t, vocab["stoi"][UNK]) for t in toks]
    return ids

class Seq2SeqDataset(Dataset):
    def __init__(self, pairs: List[Tuple[str, str]]):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        en, es = self.pairs[idx]
        src_ids = encode(en, src_vocab, add_bos_eos=False, max_len=MAX_LEN_SRC) + [src_vocab["stoi"][EOS]]
        tgt_ids = encode(es, tgt_vocab, add_bos_eos=True, max_len=MAX_LEN_TGT)
        return src_ids, tgt_ids

def collate_fn(batch):
    src_seqs, tgt_seqs = zip(*batch)
    src_lens = torch.tensor([len(s) for s in src_seqs], dtype=torch.long)

    max_src = max(len(s) for s in src_seqs)
    max_tgt = max(len(t) for t in tgt_seqs)

    src = torch.full((len(batch), max_src), pad_id_src, dtype=torch.long)
    tgt = torch.full((len(batch), max_tgt), pad_id_tgt, dtype=torch.long)

    for i, (s, t) in enumerate(zip(src_seqs, tgt_seqs)):
        src[i, :len(s)] = torch.tensor(s, dtype=torch.long)
        tgt[i, :len(t)] = torch.tensor(t, dtype=torch.long)

    tgt_in = tgt[:, :-1].contiguous()
    tgt_out = tgt[:, 1:].contiguous()

    return {"src": src, "src_lens": src_lens, "tgt_in": tgt_in, "tgt_out": tgt_out}

train_dl = DataLoader(Seq2SeqDataset(train_pairs), batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, collate_fn=collate_fn, drop_last=True)
val_dl   = DataLoader(Seq2SeqDataset(val_pairs),   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)
test_dl  = DataLoader(Seq2SeqDataset(test_pairs),  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)

print("SRC vocab:", len(src_vocab["itos"]), "| TGT vocab:", len(tgt_vocab["itos"]))

SRC vocab: 2279 | TGT vocab: 3052


## Baseline Seq2Seq

In [ ]:
enc = models.EncoderLSTM(len(src_vocab["itos"]), EMB_SRC, HIDDEN, NUM_LAYERS, DROPOUT, pad_id_src)
dec = models.DecoderLSTM(len(tgt_vocab["itos"]), EMB_TGT, HIDDEN, NUM_LAYERS, DROPOUT, pad_id_tgt)
model_base = models.Seq2Seq(enc, dec).to(DEVICE)

print("Baseline params:", sum(p.numel() for p in model_base.parameters())/1e6, "M")

Baseline params: 6.084332 M


## Train baseline (teacher forcing)

In [28]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_id_tgt)
optimizer = torch.optim.Adam(model_base.parameters(), lr=1e-3)

def train_one_epoch(model: nn.Module, dl) -> float:
    model.train()
    total = 0.0
    steps = 0
    for batch in dl:
        src = batch["src"].to(DEVICE)
        src_lens = batch["src_lens"].to(DEVICE)
        tgt_in = batch["tgt_in"].to(DEVICE)
        tgt_out = batch["tgt_out"].to(DEVICE)

        logits = model(src, src_lens, tgt_in)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total += float(loss.item())
        steps += 1
    return total / max(steps, 1)

@torch.no_grad()
def eval_one_epoch(model: nn.Module, dl) -> float:
    model.eval()
    total = 0.0
    steps = 0
    for batch in dl:
        src = batch["src"].to(DEVICE)
        src_lens = batch["src_lens"].to(DEVICE)
        tgt_in = batch["tgt_in"].to(DEVICE)
        tgt_out = batch["tgt_out"].to(DEVICE)

        logits = model(src, src_lens, tgt_in)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
        total += float(loss.item())
        steps += 1
    return total / max(steps, 1)

BASE_BEST = MODELS_DIR / "baseline_no_attention_best.pt"
BASE_LAST = MODELS_DIR / "baseline_no_attention_last.pt"

In [29]:
best_val = float("inf")
base_val_losses = []
base_epoch_times = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.perf_counter()
    train_loss = train_one_epoch(model_base, train_dl)
    val_loss = eval_one_epoch(model_base, val_dl)
    t1 = time.perf_counter()

    base_val_losses.append(val_loss)
    base_epoch_times.append(t1 - t0)

    print(f"[Baseline] Epoch {epoch:02d} | train={train_loss:.4f} | val={val_loss:.4f} | time={t1-t0:.2f}s")

    torch.save({
        "epoch": epoch,
        "model_state": model_base.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "val_loss": val_loss,
        "config": {"EMB_SRC": EMB_SRC, "EMB_TGT": EMB_TGT, "HIDDEN": HIDDEN, "NUM_LAYERS": NUM_LAYERS, "DROPOUT": DROPOUT}
    }, BASE_LAST)

    if val_loss < best_val:
        best_val = val_loss
        torch.save(torch.load(BASE_LAST, map_location="cpu"), BASE_BEST)
        print("✅ Saved best:", BASE_BEST)

print("Baseline best val:", best_val)

[Baseline] Epoch 01 | train=4.7519 | val=3.8421 | time=30.98s
✅ Saved best: models/baseline_no_attention_best.pt
[Baseline] Epoch 02 | train=3.6957 | val=3.2921 | time=29.86s
✅ Saved best: models/baseline_no_attention_best.pt
[Baseline] Epoch 03 | train=3.1150 | val=2.9749 | time=30.03s
✅ Saved best: models/baseline_no_attention_best.pt
[Baseline] Epoch 04 | train=2.6642 | val=2.7877 | time=30.24s
✅ Saved best: models/baseline_no_attention_best.pt
[Baseline] Epoch 05 | train=2.2719 | val=2.6775 | time=29.95s
✅ Saved best: models/baseline_no_attention_best.pt
[Baseline] Epoch 06 | train=1.9245 | val=2.5969 | time=30.07s
✅ Saved best: models/baseline_no_attention_best.pt
[Baseline] Epoch 07 | train=1.6003 | val=2.5636 | time=29.21s
✅ Saved best: models/baseline_no_attention_best.pt
[Baseline] Epoch 08 | train=1.3135 | val=2.5352 | time=28.41s
✅ Saved best: models/baseline_no_attention_best.pt
[Baseline] Epoch 09 | train=1.0586 | val=2.5250 | time=33.15s
✅ Saved best: models/baseline_no_a

## Evaluate baseline BLEU

In [ ]:
bleu = sacrebleu.metrics.BLEU()

hyps, refs = [], []
t0 = time.perf_counter()
for batch in test_dl:
    src = batch["src"].to(DEVICE)
    src_lens = batch["src_lens"].to(DEVICE)
    tgt_out = batch["tgt_out"].cpu()

    pred_ids = models.greedy_decode_baseline(model_base, src, src_lens, bos_id_tgt, eos_id_tgt, max_len=60).cpu()
    hyps.extend(batch_ids_to_sentences(pred_ids, tgt_vocab["itos"]))
    refs.extend(batch_ids_to_sentences(tgt_out, tgt_vocab["itos"]))

t1 = time.perf_counter()
score = bleu.corpus_score(hyps, [refs])
print("Baseline BLEU:", score, "| time:", f"{t1-t0:.2f}s")

pred_path = OUT_DIR / "baseline_predictions.tsv"
with open(pred_path, "w", encoding="utf-8") as f:
    for r, h in zip(refs, hyps):
        f.write(r + "\t" + h + "\n")

metrics_path = OUT_DIR / "baseline_bleu.json"
signature = getattr(score, "signature", None)
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump({
        "bleu": float(score.score),
        "signature": str(signature) if signature is not None else "N/A",
        "num_sentences": len(hyps),
    }, f, indent=2)

print("Saved:", pred_path, metrics_path)

Baseline BLEU: BLEU = 23.48 54.2/28.5/18.7/10.5 (BP = 1.000 ratio = 1.005 hyp_len = 8661 ref_len = 8614) | time: 8.21s
Saved: outputs/baseline_predictions.tsv outputs/baseline_bleu.json


In [31]:
def save_run_log(path: Path, run_name: str, epoch_losses: List[float], epoch_times: List[float], extra: Dict[str, Any]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump({
            "run_name": run_name,
            "epoch_losses": [float(x) for x in epoch_losses],
            "epoch_times": [float(x) for x in epoch_times],
            "extra": extra
        }, f, indent=2)

## Bahdanau (Additive) Attention model

In [ ]:
attn = models.BahdanauAttention(enc_hidden_dim=HIDDEN, dec_hidden_dim=HIDDEN, attn_dim=256)
enc2 = models.EncoderLSTMWithOutputs(len(src_vocab["itos"]), EMB_SRC, HIDDEN, NUM_LAYERS, DROPOUT, pad_id_src)
dec2 = models.AttnDecoderBahdanau(len(tgt_vocab["itos"]), EMB_TGT, HIDDEN, attn, NUM_LAYERS, DROPOUT, pad_id_tgt)

model_bah = models.Seq2SeqBahdanau(enc2, dec2, pad_id_src=pad_id_src).to(DEVICE)
print("Bahdanau params:", sum(p.numel() for p in model_bah.parameters())/1e6, "M")

Bahdanau params: 8.957932 M


## Train Bahdanau attention + save checkpoint

In [34]:
criterion_attn = nn.CrossEntropyLoss(ignore_index=pad_id_tgt)
optimizer_attn = torch.optim.Adam(model_bah.parameters(), lr=1e-3)

BAH_BEST = MODELS_DIR / "bahdanau_best.pt"
BAH_LAST = MODELS_DIR / "bahdanau_last.pt"

In [35]:
def train_one_epoch_attn(model: nn.Module, dl) -> float:
    model.train()
    total, steps = 0.0, 0
    for batch in dl:
        src = batch["src"].to(DEVICE)
        src_lens = batch["src_lens"].to(DEVICE)
        tgt_in = batch["tgt_in"].to(DEVICE)
        tgt_out = batch["tgt_out"].to(DEVICE)

        logits, _attn = model(src, src_lens, tgt_in)
        loss = criterion_attn(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

        optimizer_attn.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer_attn.step()

        total += float(loss.item()); steps += 1
    return total / max(steps, 1)

@torch.no_grad()
def eval_one_epoch_attn(model: nn.Module, dl) -> float:
    model.eval()
    total, steps = 0.0, 0
    for batch in dl:
        src = batch["src"].to(DEVICE)
        src_lens = batch["src_lens"].to(DEVICE)
        tgt_in = batch["tgt_in"].to(DEVICE)
        tgt_out = batch["tgt_out"].to(DEVICE)

        logits, _attn = model(src, src_lens, tgt_in)
        loss = criterion_attn(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

        total += float(loss.item()); steps += 1
    return total / max(steps, 1)

In [ ]:
best_val = float("inf")
bah_val_losses = []
bah_epoch_times = []

attn_cache = []
ATTN_SAVE = Path("outputs/bahdanau_attn_samples.pt")

for epoch in range(1, EPOCHS + 1):
    t0 = time.perf_counter()
    train_loss = train_one_epoch_attn(model_bah, train_dl)
    val_loss = eval_one_epoch_attn(model_bah, val_dl)
    t1 = time.perf_counter()

    bah_val_losses.append(val_loss)
    bah_epoch_times.append(t1 - t0)

    print(f"[Bahdanau] Epoch {epoch:02d} | train={train_loss:.4f} | val={val_loss:.4f} | time={t1-t0:.2f}s")

    batch = next(iter(val_dl))
    src = batch["src"].to(DEVICE); src_lens = batch["src_lens"].to(DEVICE)
    pred_ids, attn_mat = models.greedy_decode_bahdanau(model_bah, src[:1], src_lens[:1], bos_id_tgt, eos_id_tgt, max_len=40, return_attn=True)
    if attn_mat is not None:
        attn_cache.append({
            "epoch": epoch,
            "src_ids": batch["src"][0].cpu(),
            "pred_ids": pred_ids[0].cpu(),
            "attn": attn_mat[0].cpu(),
        })

    torch.save({
        "epoch": epoch,
        "model_state": model_bah.state_dict(),
        "optimizer_state": optimizer_attn.state_dict(),
        "val_loss": val_loss,
        "config": {"EMB_SRC": EMB_SRC, "EMB_TGT": EMB_TGT, "HIDDEN": HIDDEN, "NUM_LAYERS": NUM_LAYERS, "DROPOUT": DROPOUT, "attention": "bahdanau"}
    }, BAH_LAST)

    if val_loss < best_val:
        best_val = val_loss
        torch.save(torch.load(BAH_LAST, map_location="cpu"), BAH_BEST)
        print("✅ Saved best:", BAH_BEST)

Path("outputs").mkdir(parents=True, exist_ok=True)
torch.save(attn_cache, ATTN_SAVE)
print("Saved attention samples:", ATTN_SAVE)
print("Bahdanau best val:", best_val)

[Bahdanau] Epoch 01 | train=4.5357 | val=3.4507 | time=79.94s
✅ Saved best: models/bahdanau_best.pt
[Bahdanau] Epoch 02 | train=3.1231 | val=2.6923 | time=79.79s
✅ Saved best: models/bahdanau_best.pt
[Bahdanau] Epoch 03 | train=2.1894 | val=2.3289 | time=81.44s
✅ Saved best: models/bahdanau_best.pt
[Bahdanau] Epoch 04 | train=1.5001 | val=2.1600 | time=81.04s
✅ Saved best: models/bahdanau_best.pt
[Bahdanau] Epoch 05 | train=1.0095 | val=2.1009 | time=80.83s
✅ Saved best: models/bahdanau_best.pt
[Bahdanau] Epoch 06 | train=0.6881 | val=2.0884 | time=80.66s
✅ Saved best: models/bahdanau_best.pt
[Bahdanau] Epoch 07 | train=0.4686 | val=2.0965 | time=80.73s
[Bahdanau] Epoch 08 | train=0.3270 | val=2.1326 | time=80.61s
[Bahdanau] Epoch 09 | train=0.2272 | val=2.1989 | time=80.14s
[Bahdanau] Epoch 10 | train=0.1624 | val=2.2204 | time=80.80s
[Bahdanau] Epoch 11 | train=0.1206 | val=2.2651 | time=78.44s
[Bahdanau] Epoch 12 | train=0.0886 | val=2.3032 | time=78.84s
[Bahdanau] Epoch 13 | train=

## Evaluate Bahdanau BLEU

In [ ]:
bleu = sacrebleu.metrics.BLEU()
hyps, refs = [], []

t0 = time.perf_counter()
for batch in test_dl:
    src = batch["src"].to(DEVICE)
    src_lens = batch["src_lens"].to(DEVICE)
    tgt_out = batch["tgt_out"].cpu()

    pred_ids, _ = models.greedy_decode_bahdanau(model_bah, src, src_lens, bos_id_tgt, eos_id_tgt, max_len=60, return_attn=False)
    pred_ids = pred_ids.cpu()

    hyps.extend(batch_ids_to_sentences(pred_ids, tgt_vocab["itos"]))
    refs.extend(batch_ids_to_sentences(tgt_out, tgt_vocab["itos"]))

t1 = time.perf_counter()
score = bleu.corpus_score(hyps, [refs])
print("Bahdanau BLEU:", score, "| time:", f"{t1-t0:.2f}s")

pred_path = OUT_DIR / "bahdanau_predictions.tsv"
with open(pred_path, "w", encoding="utf-8") as f:
    for r, h in zip(refs, hyps):
        f.write(r + "\t" + h + "\n")

metrics_path = OUT_DIR / "bahdanau_bleu.json"
signature = getattr(score, "signature", None)
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump({
        "bleu": float(score.score),
        "signature": str(signature) if signature is not None else "N/A",
        "num_sentences": len(hyps),
    }, f, indent=2)

print("Saved:", pred_path, metrics_path)


Bahdanau BLEU: BLEU = 30.97 61.2/37.7/25.7/15.5 (BP = 1.000 ratio = 1.020 hyp_len = 8786 ref_len = 8614) | time: 13.13s
Saved: outputs/bahdanau_predictions.tsv outputs/bahdanau_bleu.json


## Luong Attention model

In [ ]:
attn = models.LuongAttention(hidden_dim=HIDDEN, method="general")
enc2 = models.EncoderLSTMWithOutputs(len(src_vocab["itos"]), EMB_SRC, HIDDEN, NUM_LAYERS, DROPOUT, pad_id_src)
dec2 = models.AttnDecoderLuong(len(tgt_vocab["itos"]), EMB_TGT, HIDDEN, attn, NUM_LAYERS, DROPOUT, pad_id_tgt)

model_luong = models.Seq2SeqLuong(enc2, dec2, pad_id_src=pad_id_src).to(DEVICE)
print("Luong params:", sum(p.numel() for p in model_luong.parameters())/1e6, "M")

Luong params: 6.871276 M


## Train Luong attention + save checkpoint

In [40]:
criterion_attn = nn.CrossEntropyLoss(ignore_index=pad_id_tgt)
optimizer_attn = torch.optim.Adam(model_luong.parameters(), lr=1e-3)

LUONG_BEST = MODELS_DIR / "luong_best.pt"
LUONG_LAST = MODELS_DIR / "luong_last.pt"

In [ ]:
best_val = float("inf")
luong_val_losses = []
luong_epoch_times = []

attn_cache = []
ATTN_SAVE = Path("outputs/luong_attn_samples.pt")

for epoch in range(1, EPOCHS + 1):
    t0 = time.perf_counter()
    train_loss = train_one_epoch_attn(model_luong, train_dl)
    val_loss = eval_one_epoch_attn(model_luong, val_dl)
    t1 = time.perf_counter()

    luong_val_losses.append(val_loss)
    luong_epoch_times.append(t1 - t0)

    print(f"[Luong] Epoch {epoch:02d} | train={train_loss:.4f} | val={val_loss:.4f} | time={t1-t0:.2f}s")

    batch = next(iter(val_dl))
    src = batch["src"].to(DEVICE); src_lens = batch["src_lens"].to(DEVICE)
    pred_ids, attn_mat = models.greedy_decode_luong(model_luong, src[:1], src_lens[:1], bos_id_tgt, eos_id_tgt, max_len=40, return_attn=True)
    if attn_mat is not None:
        attn_cache.append({
            "epoch": epoch,
            "src_ids": batch["src"][0].cpu(),
            "pred_ids": pred_ids[0].cpu(),
            "attn": attn_mat[0].cpu(),
        })

    torch.save({
        "epoch": epoch,
        "model_state": model_luong.state_dict(),
        "optimizer_state": optimizer_attn.state_dict(),
        "val_loss": val_loss,
        "config": {"EMB_SRC": EMB_SRC, "EMB_TGT": EMB_TGT, "HIDDEN": HIDDEN, "NUM_LAYERS": NUM_LAYERS, "DROPOUT": DROPOUT, "attention": "luong"}
    }, LUONG_LAST)

    if val_loss < best_val:
        best_val = val_loss
        torch.save(torch.load(LUONG_LAST, map_location="cpu"), LUONG_BEST)
        print("✅ Saved best:", LUONG_BEST)

Path("outputs").mkdir(parents=True, exist_ok=True)
torch.save(attn_cache, ATTN_SAVE)
print("Saved attention samples:", ATTN_SAVE)
print("Luong best val:", best_val)

[Luong] Epoch 01 | train=4.6620 | val=3.6838 | time=73.94s
✅ Saved best: models/luong_best.pt
[Luong] Epoch 02 | train=3.5656 | val=3.1250 | time=74.90s
✅ Saved best: models/luong_best.pt
[Luong] Epoch 03 | train=2.9480 | val=2.8203 | time=74.91s
✅ Saved best: models/luong_best.pt
[Luong] Epoch 04 | train=2.4200 | val=2.6108 | time=73.98s
✅ Saved best: models/luong_best.pt
[Luong] Epoch 05 | train=1.9512 | val=2.4944 | time=74.30s
✅ Saved best: models/luong_best.pt
[Luong] Epoch 06 | train=1.5323 | val=2.4240 | time=74.89s
✅ Saved best: models/luong_best.pt
[Luong] Epoch 07 | train=1.1590 | val=2.4309 | time=75.18s
[Luong] Epoch 08 | train=0.8419 | val=2.4695 | time=73.93s
[Luong] Epoch 09 | train=0.5874 | val=2.5047 | time=73.46s
[Luong] Epoch 10 | train=0.4124 | val=2.5679 | time=74.54s
[Luong] Epoch 11 | train=0.2800 | val=2.6291 | time=73.22s
[Luong] Epoch 12 | train=0.1983 | val=2.6962 | time=71.66s
[Luong] Epoch 13 | train=0.1441 | val=2.7558 | time=71.85s
[Luong] Epoch 14 | trai

## Evaluate Luong BLEU

In [ ]:
bleu = sacrebleu.metrics.BLEU()
hyps, refs = [], []

t0 = time.perf_counter()
for batch in test_dl:
    src = batch["src"].to(DEVICE)
    src_lens = batch["src_lens"].to(DEVICE)
    tgt_out = batch["tgt_out"].cpu()

    pred_ids, _ = models.greedy_decode_luong(model_luong, src, src_lens, bos_id_tgt, eos_id_tgt, max_len=60, return_attn=False)
    pred_ids = pred_ids.cpu()

    hyps.extend(batch_ids_to_sentences(pred_ids, tgt_vocab["itos"]))
    refs.extend(batch_ids_to_sentences(tgt_out, tgt_vocab["itos"]))

t1 = time.perf_counter()
score = bleu.corpus_score(hyps, [refs])
print("Luong BLEU:", score, "| time:", f"{t1-t0:.2f}s")

pred_path = OUT_DIR / "luong_predictions.tsv"
with open(pred_path, "w", encoding="utf-8") as f:
    for r, h in zip(refs, hyps):
        f.write(r + "\t" + h + "\n")

metrics_path = OUT_DIR / "luong_bleu.json"
signature = getattr(score, "signature", None)
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump({
        "bleu": float(score.score),
        "signature": str(signature) if signature is not None else "N/A",
        "num_sentences": len(hyps),
    }, f, indent=2)

print("Saved:", pred_path, metrics_path)


Luong BLEU: BLEU = 23.42 56.1/30.6/19.7/10.9 (BP = 0.951 ratio = 0.952 hyp_len = 8203 ref_len = 8614) | time: 11.95s
Saved: outputs/luong_predictions.tsv outputs/luong_bleu.json


## Save predictions metrics

In [43]:
save_run_log(RUNS_DIR / "baseline.json", "baseline_no_attention", base_val_losses, base_epoch_times,
             {"epochs": EPOCHS, "device": DEVICE, "bleu_file": "outputs/baseline_bleu.json"})
save_run_log(RUNS_DIR / "bahdanau.json", "bahdanau_attention", bah_val_losses, bah_epoch_times,
             {"epochs": EPOCHS, "device": DEVICE, "bleu_file": "outputs/bahdanau_bleu.json"})
save_run_log(RUNS_DIR / "luong.json", "luong_attention", luong_val_losses, luong_epoch_times,
             {"epochs": EPOCHS, "device": DEVICE, "bleu_file": "outputs/luong_bleu.json"})

print("Saved run logs to:", RUNS_DIR)

Saved run logs to: runs
